In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate
from tensorflow.keras.models import Model
import numpy as np

In [71]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


In [23]:
Conj_val=np.load('Features/conjunctiva_val_features.npy')
Text_val=np.load('Features/text_val.npy')

In [24]:
Conjunctiva_features = np.load("Features/Conjunctiva_train_features.npy")
Textual_features = np.load("Features/text_train.npy")


In [25]:
Conjunctiva_features = tf.convert_to_tensor(Conjunctiva_features, dtype=tf.float32)
Textual_features = tf.convert_to_tensor(Textual_features, dtype=tf.float32)
Conj_val = tf.convert_to_tensor(Conj_val, dtype=tf.float32)
Text_val = tf.convert_to_tensor(Text_val, dtype=tf.float32)

In [26]:
Conjunctiva_features = tf.expand_dims(Conjunctiva_features, axis=1)
Textual_features = tf.expand_dims(Textual_features, axis=1)
Conj_val = tf.expand_dims(Conj_val, axis=1)
Text_val = tf.expand_dims (Text_val, axis =1)

# Self Attention

In [27]:
def self_attention_block(x, num_heads=4):
    d_model = x.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(x, x)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(x + attn)
    return out

In [28]:
conj_input = Input(shape=(1, Conjunctiva_features.shape[-1]), name="Conjunctiva_Input")
text_input = Input(shape=(1, Textual_features.shape[-1]), name="Text_Input")
conj_self = self_attention_block(conj_input)
text_self = self_attention_block(text_input)

# Cross Attention

In [29]:
def cross_attention_block(query, key_value, num_heads=4):
    d_model = query.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(query, key_value)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(query + attn)
    return out


In [30]:
text_cross_conj = cross_attention_block(text_self, conj_self)

In [ ]:
from tensorflow.keras.layers import Flatten

conj_flat = Flatten()(conj_self)
text_flat = Flatten()(text_self)
text_cross_flat = Flatten()(text_cross_conj)

fusion_vector = Concatenate()([conj_flat, text_flat, text_cross_flat])


In [32]:
x = Dense(512, activation='relu')(fusion_vector)
x = Dropout(0.5)(x)

output_class = Dense(1, activation='sigmoid', name="Anemia_Class")(x)  # classification
output_hb = Dense(1, activation='linear', name="Hemoglobin")(x)        # regression


In [33]:
final_model = Model(
    inputs=[conj_input, text_input],
    outputs=[output_class, output_hb]
)

final_model.compile(
    optimizer='adam',
    loss={'Anemia_Class':'binary_crossentropy', 'Hemoglobin':'mse'},
    metrics={'Anemia_Class':'accuracy', 'Hemoglobin':'mae'}
)

final_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Conjunctiva_Input   │ (None, 1, 1024)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_Input          │ (None, 1, 32)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 1024)   │  4,198,400 │ Conjunctiva_Inpu… │
│ (MultiHeadAttentio… │                   │            │ Conjunctiva_Inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 32)     │      4,224 │ Text_Input[0][0], │
│ (MultiHeadAttentio… │                   │            │ Text_Input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 1, 1024)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 1, 32)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 1, 1024)   │          0 │ Conjunctiva_Inpu… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 1, 32)     │          0 │ Text_Input[0][0], │
│                     │                   │            │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 1024)   │      2,048 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 32)     │         64 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 32)     │     67,712 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 1, 32)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 32)     │         64 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 1024)      │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 32)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_5 (Flatten) │ (None, 32)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 1088)      │          0 │ flatten_3[0][0],  │
│ (Concatenate)       │                   │            │ flatten_4[0][0],

 Total params: 4,831,106 (18.43 MB)

 Trainable params: 4,831,106 (18.43 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
y_train_class = np.load("y_train_class.npy")
y_train_hb = np.load("y_train_hb.npy")

In [35]:
y_val_class = np.load("y_val_class.npy")
y_val_hb = np.load("y_val_hb.npy")

In [36]:
print("Conjunctiva features shape:", Conjunctiva_features.shape)
print("Textual features shape:", Textual_features.shape)
print("y_train_class shape:", y_train_class.shape)
print("y_train_hb shape:", y_train_hb.shape)
print("Conj_val shape:", Conj_val.shape)
print("Text_val shape:", Text_val.shape)
print("y_val_class shape:", y_val_class.shape)
print("y_val_hb shape:", y_val_hb.shape)


Conjunctiva features shape: (496, 1, 1024)
Textual features shape: (496, 1, 32)
y_train_class shape: (496,)
y_train_hb shape: (496,)
Conj_val shape: (105, 1, 1024)
Text_val shape: (105, 1, 32)
y_val_class shape: (105,)
y_val_hb shape: (105,)


In [ ]:
# Compute sample weights for the classification output
from sklearn.utils import class_weight
import numpy as np

# y_train_class = 0/1
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_class),
    y=y_train_class
)

sample_weights = np.array([class_weights[int(c)] for c in y_train_class])

sample_weight_dict = {
    final_model.get_layer("Anemia_Class").output: sample_weights,
    final_model.get_layer("Hemoglobin").output: np.ones_like(y_train_hb)
}

final_model.fit(
    [Conjunctiva_features, Textual_features],
    [y_train_class, y_train_hb],
    validation_data=([Conj_val, Text_val], [y_val_class, y_val_hb]),
    epochs=30,
    batch_size=16,
    sample_weight=[sample_weights, None]  # first for classification, second for regression
)



Epoch 1/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - Anemia_Class_accuracy: 0.5302 - Anemia_Class_loss: 0.7240 - Hemoglobin_loss: 6.8902 - Hemoglobin_mae: 2.0789 - loss: 7.6142 - val_Anemia_Class_accuracy: 0.3619 - val_Anemia_Class_loss: 0.7129 - val_Hemoglobin_loss: 6.3991 - val_Hemoglobin_mae: 1.9378 - val_loss: 6.9839
Epoch 2/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - Anemia_Class_accuracy: 0.4879 - Anemia_Class_loss: 0.7302 - Hemoglobin_loss: 6.5930 - Hemoglobin_mae: 2.0586 - loss: 7.3232 - val_Anemia_Class_accuracy: 0.3619 - val_Anemia_Class_loss: 0.8374 - val_Hemoglobin_loss: 10.9674 - val_Hemoglobin_mae: 2.7575 - val_loss: 12.1820
Epoch 3/30
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - Anemia_Class_accuracy: 0.4899 - Anemia_Class_loss: 0.7139 - Hemoglobin_loss: 6.5660 - Hemoglobin_mae: 2.0031 - loss: 7.2799 - val_Anemia_Class_accuracy: 0.6381 - val_Anemia_Class_loss: 0.6774 - val_Hemoglobin_loss: 6.8726 - val_Hemoglobin_mae: 2.0425 - val_loss: 7.4832
Epoch 4/30
31/31 ━━━━━━━━━━━━

In [50]:
# Load test features
Conj_test = np.load("Features/conjunctiva_test_features.npy")
Text_test = np.load("Features/text_test.npy")

# Convert to tensor
Conj_test = tf.convert_to_tensor(Conj_test, dtype=tf.float32)
Text_test = tf.convert_to_tensor(Text_test, dtype=tf.float32)

# Expand dims for attention layer
Conj_test = tf.expand_dims(Conj_test, axis=1)
Text_test = tf.expand_dims(Text_test, axis=1)

# Load test labels
y_test_class = np.load("y_test_class.npy")
y_test_hb = np.load("y_test_hb.npy")


In [51]:
results = final_model.evaluate(
    [Conj_test, Text_test],
    [y_test_class, y_test_hb],
    batch_size=16
)

print("Test Results:", dict(zip(final_model.metrics_names, results)))


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - Anemia_Class_accuracy: 0.3945 - Anemia_Class_loss: 0.7848 - Hemoglobin_loss: 8.2449 - Hemoglobin_mae: 2.3653 - loss: 9.1051
Test Results: {'loss': 9.105094909667969, 'compile_metrics': 0.7847974896430969, 'Anemia_Class_loss': 8.244929313659668, 'Hemoglobin_loss': 0.39449542760849}


In [ ]:

pred_class, pred_hb = final_model.predict([Conj_test, Text_test])
pred_class_label = (pred_class > 0.5).astype(int)
print(f"{'Index':<6} {'Actual Class':<12} {'Pred Class':<12} {'Actual Hb':<10} {'Pred Hb':<10}")
for i in range(20):
    print(f"{i:<6} {y_test_class[i]:<12} {pred_class_label[i][0]:<12} {y_test_hb[i]:<10.2f} {pred_hb[i][0]:<10.2f}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Index  Actual Class Pred Class   Actual Hb  Pred Hb   
0      1.0          1            12.40      10.84     
1      1.0          1            9.90       10.93     
2      1.0          0            9.90       12.36     
3      0.0          0            12.90      12.77     
4      1.0          0            8.20       12.33     
5      1.0          0            8.30       12.58     
6      1.0          0            8.30       11.88     
7      1.0          0            8.30       12.25     
8      1.0          1            8.30       9.99      
9      0.0          0            11.90      11.17     
10     0.0          0            12.70      11.13     
11     0.0          1            12.40      10.23     
12     0.0          0            11.00      11.38     
13     1.0          0            10.10      11.61     
14     1.0          0            10.70      12.06     
15     0.0          0            12.90      13.37     
16     1.0          0      

# 2

In [55]:
def cross_attention_block(query, key_value, num_heads=4):
    d_model = query.shape[-1]
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(query, key_value)
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(query + attn)
    return out

# Text attends to Conjunctiva
text_cross_conj = cross_attention_block(text_self, conj_self)
# Conjunctiva attends to Text
conj_cross_text = cross_attention_block(conj_self, text_self)


In [56]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate, GlobalAveragePooling1D
from tensorflow.keras.models import Model

In [57]:
conj_pool = GlobalAveragePooling1D()(conj_self)
text_pool = GlobalAveragePooling1D()(text_self)
text_cross_pool = GlobalAveragePooling1D()(text_cross_conj)
conj_cross_pool = GlobalAveragePooling1D()(conj_cross_text)


In [ ]:
# Shared fusion vector
fusion_vector = Concatenate()([conj_pool, text_pool, text_cross_pool, conj_cross_pool])

# Classification branch
x_class = Dense(512, activation='relu')(fusion_vector)
x_class = Dropout(0.5)(x_class)
x_class = Dense(256, activation='relu')(x_class)
output_class = Dense(1, activation='sigmoid', name="Anemia_Class")(x_class)

# Regression branch
x_hb = Dense(256, activation='relu')(fusion_vector)
x_hb = Dropout(0.3)(x_hb)
output_hb = Dense(1, activation='linear', name="Hemoglobin")(x_hb)



In [65]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input

# Assuming conj_input and text_input already exist
final_model = Model(
    inputs=[conj_input, text_input],
    outputs=[output_class, output_hb]
)


In [66]:
final_model.compile(
    optimizer='adam',
    loss={
        "Anemia_Class": "binary_crossentropy",
        "Hemoglobin": "mse"
    },
    metrics={
        "Anemia_Class": "accuracy",
        "Hemoglobin": "mae"
    },
    loss_weights={
        "Anemia_Class": 1.0,   # focus more on classification
        "Hemoglobin": 0.5      # optional: regression is secondary
    }
)


In [67]:
history = final_model.fit(
    [Conjunctiva_features, Textual_features],  # inputs
    {"Anemia_Class": y_train_class, "Hemoglobin": y_train_hb},  # outputs
    validation_data=(
        [Conj_val, Text_val],
        {"Anemia_Class": y_val_class, "Hemoglobin": y_val_hb}
    ),
    epochs=40,
    batch_size=16
)


Epoch 1/40
31/31 ━━━━━━━━━━━━━━━━━━━━ 16s 155ms/step - Anemia_Class_accuracy: 0.5948 - Anemia_Class_loss: 0.8649 - Hemoglobin_loss: 15.0671 - Hemoglobin_mae: 2.7754 - loss: 8.3985 - val_Anemia_Class_accuracy: 0.3619 - val_Anemia_Class_loss: 1.1335 - val_Hemoglobin_loss: 11.3330 - val_Hemoglobin_mae: 2.8006 - val_loss: 7.0208
Epoch 2/40
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 135ms/step - Anemia_Class_accuracy: 0.6250 - Anemia_Class_loss: 0.7278 - Hemoglobin_loss: 5.9564 - Hemoglobin_mae: 1.9497 - loss: 3.7060 - val_Anemia_Class_accuracy: 0.6095 - val_Anemia_Class_loss: 0.6711 - val_Hemoglobin_loss: 9.9076 - val_Hemoglobin_mae: 2.5797 - val_loss: 5.7357
Epoch 3/40
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 125ms/step - Anemia_Class_accuracy: 0.6472 - Anemia_Class_loss: 0.6560 - Hemoglobin_loss: 5.3974 - Hemoglobin_mae: 1.8134 - loss: 3.3547 - val_Anemia_Class_accuracy: 0.6476 - val_Anemia_Class_loss: 0.7213 - val_Hemoglobin_loss: 16.3647 - val_Hemoglobin_mae: 3.4397 - val_loss: 9.0469
Epoch 4/40
31/31 ━━━━━━━

In [68]:
results = final_model.evaluate(
    [Conj_test, Text_test],
    {"Anemia_Class": y_test_class, "Hemoglobin": y_test_hb},
    batch_size=16
)

print("Test Loss (combined):", results[0])
print("Test Classification Accuracy:", results[3])  # check Keras metric order
print("Test Hemoglobin MAE:", results[4])


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - Anemia_Class_accuracy: 0.4220 - Anemia_Class_loss: 0.8204 - Hemoglobin_loss: 18.5278 - Hemoglobin_mae: 3.6079 - loss: 10.1978
Test Loss (combined): 10.197755813598633
Test Classification Accuracy: 0.4220183491706848
Test Hemoglobin MAE: 3.607914447784424


In [69]:
pred_class, pred_hb = final_model.predict([Conj_test, Text_test])

# Convert classification probabilities to 0/1
pred_class_label = (pred_class > 0.5).astype(int)

# Quick look at first 20 predictions
for i in range(20):
    print(f"Index: {i}, Actual Class: {y_test_class[i]}, Pred Class: {pred_class_label[i][0]}, "
          f"Actual Hb: {y_test_hb[i]:.2f}, Pred Hb: {pred_hb[i][0]:.2f}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 153ms/step
Index: 0, Actual Class: 1.0, Pred Class: 1, Actual Hb: 12.40, Pred Hb: 13.17
Index: 1, Actual Class: 1.0, Pred Class: 1, Actual Hb: 9.90, Pred Hb: 13.11
Index: 2, Actual Class: 1.0, Pred Class: 0, Actual Hb: 9.90, Pred Hb: 14.55
Index: 3, Actual Class: 0.0, Pred Class: 0, Actual Hb: 12.90, Pred Hb: 15.42
Index: 4, Actual Class: 1.0, Pred Class: 0, Actual Hb: 8.20, Pred Hb: 14.02
Index: 5, Actual Class: 1.0, Pred Class: 0, Actual Hb: 8.30, Pred Hb: 15.11
Index: 6, Actual Class: 1.0, Pred Class: 0, Actual Hb: 8.30, Pred Hb: 14.33
Index: 7, Actual Class: 1.0, Pred Class: 0, Actual Hb: 8.30, Pred Hb: 14.55
Index: 8, Actual Class: 1.0, Pred Class: 1, Actual Hb: 8.30, Pred Hb: 12.73
Index: 9, Actual Class: 0.0, Pred Class: 1, Actual Hb: 11.90, Pred Hb: 12.23
Index: 10, Actual Class: 0.0, Pred Class: 1, Actual Hb: 12.70, Pred Hb: 12.68
Index: 11, Actual Class: 0.0, Pred Class: 1, Actual Hb: 12.40, Pred Hb: 12.19
Index: 12, Actual Class: 0.0, Pred Class: 

In [73]:
accuracy = accuracy_score(y_test_class, pred_class_label)
precision = precision_score(y_test_class, pred_class_label)
recall = recall_score(y_test_class, pred_class_label)
f1 = f1_score(y_test_class, pred_class_label)
roc_auc = roc_auc_score(y_test_class, pred_class)  # use probabilities for ROC-AUC

print("Classification Metrics:")
print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1-Score   : {f1:.4f}")
print(f"ROC-AUC    : {roc_auc:.4f}")


Classification Metrics:
Accuracy   : 0.4220
Precision  : 0.5854
Recall     : 0.3429
F1-Score   : 0.4324
ROC-AUC    : 0.4883


In [77]:
mae = mean_absolute_error(y_test_hb, pred_hb)
mse = mean_squared_error(y_test_hb, pred_hb) 
rmse = np.sqrt(mse) 
r2 = r2_score(y_test_hb, pred_hb)

print("\nRegression Metrics:")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")



Regression Metrics:
MAE  : 3.6079
MSE  : 18.7444
RMSE : 4.3295
R2   : -2.7649


# New

In [78]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [79]:
Conjunctiva_train = np.load("Features/conjunctiva_train_features.npy")
Text_train = np.load("Features/text_train.npy")
Conjunctiva_val = np.load('Features/conjunctiva_val_features.npy')
Text_val = np.load('Features/text_val.npy')
Conjunctiva_test = np.load('Features/conjunctiva_test_features.npy')
Text_test = np.load('Features/text_test.npy')

In [80]:
y_train_class = np.load("y_train_class.npy")
y_train_hb = np.load("y_train_hb.npy")
y_val_class = np.load("y_val_class.npy")
y_val_hb = np.load("y_val_hb.npy")
y_test_class = np.load("y_test_class.npy")
y_test_hb = np.load("y_test_hb.npy")

In [81]:
print("=" * 60)
print("DATA VERIFICATION")
print("=" * 60)
print(f"Train - Conj: {Conjunctiva_train.shape}, Text: {Text_train.shape}, Labels: {y_train_class.shape}")
print(f"Val   - Conj: {Conjunctiva_val.shape}, Text: {Text_val.shape}, Labels: {y_val_class.shape}")
print(f"Test  - Conj: {Conjunctiva_test.shape}, Text: {Text_test.shape}, Labels: {y_test_class.shape}")
print(f"\nClass distribution - Train: {np.bincount(y_train_class.astype(int))}")
print(f"Class distribution - Val:   {np.bincount(y_val_class.astype(int))}")
print(f"Class distribution - Test:  {np.bincount(y_test_class.astype(int))}")
print("=" * 60)

DATA VERIFICATION
Train - Conj: (496, 1024), Text: (496, 32), Labels: (496,)
Val   - Conj: (105, 1024), Text: (105, 32), Labels: (105,)
Test  - Conj: (109, 1024), Text: (109, 32), Labels: (109,)

Class distribution - Train: [185 311]
Class distribution - Val:   [38 67]
Class distribution - Test:  [39 70]


In [82]:
assert Conjunctiva_train.shape[0] == Text_train.shape[0] == y_train_class.shape[0], "Train size mismatch!"
assert Conjunctiva_val.shape[0] == Text_val.shape[0] == y_val_class.shape[0], "Val size mismatch!"
assert Conjunctiva_test.shape[0] == Text_test.shape[0] == y_test_class.shape[0], "Test size mismatch!"

In [83]:
scaler_conj = StandardScaler()
scaler_text = StandardScaler()

Conjunctiva_train = scaler_conj.fit_transform(Conjunctiva_train)
Conjunctiva_val = scaler_conj.transform(Conjunctiva_val)
Conjunctiva_test = scaler_conj.transform(Conjunctiva_test)

Text_train = scaler_text.fit_transform(Text_train)
Text_val = scaler_text.transform(Text_val)
Text_test = scaler_text.transform(Text_test)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate, BatchNormalization, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

Conjunctiva_train = np.load("Features/conjunctiva_train_features.npy")
Text_train = np.load("Features/text_train.npy")
Conjunctiva_val = np.load('Features/conjunctiva_val_features.npy')
Text_val = np.load('Features/text_val.npy')
Conjunctiva_test = np.load('Features/conjunctiva_test_features.npy')
Text_test = np.load('Features/text_test.npy')

y_train_class = np.load("y_train_class.npy")
y_train_hb = np.load("y_train_hb.npy")
y_val_class = np.load("y_val_class.npy")
y_val_hb = np.load("y_val_hb.npy")
y_test_class = np.load("y_test_class.npy")
y_test_hb = np.load("y_test_hb.npy")

print("=" * 70)
print("DATA VERIFICATION")
print("=" * 70)
print(f"Train - Conj: {Conjunctiva_train.shape}, Text: {Text_train.shape}")
print(f"Val   - Conj: {Conjunctiva_val.shape}, Text: {Text_val.shape}")
print(f"Test  - Conj: {Conjunctiva_test.shape}, Text: {Text_test.shape}")
print(f"Class distribution - Train: {np.bincount(y_train_class.astype(int))}")
print(f"Class distribution - Val:   {np.bincount(y_val_class.astype(int))}")
print(f"Class distribution - Test:  {np.bincount(y_test_class.astype(int))}")
print("=" * 70)

# Normalize features
scaler_conj = StandardScaler()
scaler_text = StandardScaler()
scaler_hb = StandardScaler()

Conjunctiva_train = scaler_conj.fit_transform(Conjunctiva_train)
Conjunctiva_val = scaler_conj.transform(Conjunctiva_val)
Conjunctiva_test = scaler_conj.transform(Conjunctiva_test)

Text_train = scaler_text.fit_transform(Text_train)
Text_val = scaler_text.transform(Text_val)
Text_test = scaler_text.transform(Text_test)

# Normalize hemoglobin values (CRITICAL FIX!)
y_train_hb_scaled = scaler_hb.fit_transform(y_train_hb.reshape(-1, 1)).flatten()
y_val_hb_scaled = scaler_hb.transform(y_val_hb.reshape(-1, 1)).flatten()
y_test_hb_scaled = scaler_hb.transform(y_test_hb.reshape(-1, 1)).flatten()

print(f"\nHemoglobin - Original range: [{y_train_hb.min():.2f}, {y_train_hb.max():.2f}]")
print(f"Hemoglobin - Scaled range: [{y_train_hb_scaled.min():.2f}, {y_train_hb_scaled.max():.2f}]")


SHARED_DIM = 128  # Common dimension for attention

def self_attention_block(x, num_heads=4, name="self_attn"):
    """Self-attention: each modality attends to itself"""
    d_model = x.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=0.1,
        name=name
    )(x, x)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(x + attn)
    return out

def cross_attention_block(query, key_value, num_heads=4, name="cross_attn"):
    """Cross-attention: one modality attends to another"""
    d_model = query.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=0.1,
        name=name
    )(query, key_value)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(query + attn)
    return out

# Define inputs (flat features)
conj_input = Input(shape=(1024,), name="Conjunctiva_Input")
text_input = Input(shape=(32,), name="Text_Input")

# ============================================================================
# FIX #1: Project both modalities to SAME dimension before attention
# ============================================================================
print("\nProjecting features to shared dimension...")

# Conjunctiva: 1024 -> 128
conj_dense = Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(conj_input)
conj_dense = BatchNormalization()(conj_dense)
conj_dense = Dropout(0.3)(conj_dense)
conj_proj = Dense(SHARED_DIM, activation='relu')(conj_dense)
conj_proj = BatchNormalization()(conj_proj)

# Text: 32 -> 128
text_dense = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(text_input)
text_dense = BatchNormalization()(text_dense)
text_dense = Dropout(0.2)(text_dense)
text_proj = Dense(SHARED_DIM, activation='relu')(text_dense)
text_proj = BatchNormalization()(text_proj)

# Add sequence dimension for attention: (batch, 128) -> (batch, 1, 128)
conj_seq = tf.keras.layers.Reshape((1, SHARED_DIM))(conj_proj)
text_seq = tf.keras.layers.Reshape((1, SHARED_DIM))(text_proj)

print(f"After projection - Conj: {conj_seq.shape}, Text: {text_seq.shape}")

# ============================================================================
# YOUR NOVELTY: Self-Attention + Cross-Attention
# ============================================================================
print("\nApplying self-attention...")
# Each modality attends to itself
conj_self = self_attention_block(conj_seq, num_heads=4, name="conj_self_attn")
text_self = self_attention_block(text_seq, num_heads=4, name="text_self_attn")

print("Applying cross-attention...")
# Cross-modal attention (each attends to the other)
text_cross_conj = cross_attention_block(text_self, conj_self, num_heads=4, name="text_to_conj")
conj_cross_text = cross_attention_block(conj_self, text_self, num_heads=4, name="conj_to_text")

# ============================================================================
# Pool attention outputs
# ============================================================================
conj_self_pool = GlobalAveragePooling1D()(conj_self)
text_self_pool = GlobalAveragePooling1D()(text_self)
text_cross_pool = GlobalAveragePooling1D()(text_cross_conj)
conj_cross_pool = GlobalAveragePooling1D()(conj_cross_text)

# Fuse all attention outputs
fusion_vector = Concatenate()([conj_self_pool, text_self_pool, text_cross_pool, conj_cross_pool])
print(f"Fusion vector shape: {fusion_vector.shape}")

# ============================================================================
# FIX #2: Simpler fusion layers (reduce overfitting)
# ============================================================================
x_shared = Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(fusion_vector)
x_shared = BatchNormalization()(x_shared)
x_shared = Dropout(0.4)(x_shared)
x_shared = Dense(64, activation='relu')(x_shared)
x_shared = Dropout(0.3)(x_shared)

# ============================================================================
# Task-specific heads
# ============================================================================
# Classification branch
x_class = Dense(32, activation='relu')(x_shared)
x_class = Dropout(0.2)(x_class)
output_class = Dense(1, activation='sigmoid', name="Anemia_Class")(x_class)

# Regression branch
x_hb = Dense(32, activation='relu')(x_shared)
x_hb = Dropout(0.2)(x_hb)
output_hb = Dense(1, activation='linear', name="Hemoglobin")(x_hb)

# ============================================================================
# Create and compile model
# ============================================================================
model = Model(
    inputs=[conj_input, text_input],
    outputs=[output_class, output_hb]
)

# FIX #3: Better loss weighting
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={
        "Anemia_Class": "binary_crossentropy",
        "Hemoglobin": "mse"  # MSE on normalized values
    },
    metrics={
        "Anemia_Class": ["accuracy", tf.keras.metrics.AUC(name='auc')],
        "Hemoglobin": ["mae"]
    },
    loss_weights={
        "Anemia_Class": 1.0,
        "Hemoglobin": 0.1  # Much lower weight for regression
    }
)

print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
model.summary()

# ============================================================================
# STEP 3: Train with callbacks
# ============================================================================
callbacks = [
    EarlyStopping(
        monitor='val_Anemia_Class_auc',
        patience=20,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    )
]

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

history = model.fit(
    [Conjunctiva_train, Text_train],
    {"Anemia_Class": y_train_class, "Hemoglobin": y_train_hb_scaled},
    validation_data=(
        [Conjunctiva_val, Text_val],
        {"Anemia_Class": y_val_class, "Hemoglobin": y_val_hb_scaled}
    ),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

# ============================================================================
# STEP 4: Evaluate on test set
# ============================================================================
print("\n" + "=" * 70)
print("EVALUATION ON TEST SET")
print("=" * 70)

# Get predictions
pred_class_prob, pred_hb_scaled = model.predict(
    [Conjunctiva_test, Text_test], 
    verbose=0
)

# Denormalize hemoglobin predictions (CRITICAL!)
pred_hb = scaler_hb.inverse_transform(pred_hb_scaled.reshape(-1, 1)).flatten()
pred_class_label = (pred_class_prob > 0.5).astype(int).flatten()

# ============================================================================
# Classification Metrics
# ============================================================================
print("\nCLASSIFICATION METRICS:")
print("-" * 70)
acc = accuracy_score(y_test_class, pred_class_label)
prec = precision_score(y_test_class, pred_class_label, zero_division=0)
rec = recall_score(y_test_class, pred_class_label, zero_division=0)
f1 = f1_score(y_test_class, pred_class_label, zero_division=0)
auc = roc_auc_score(y_test_class, pred_class_prob)

print(f"Accuracy   : {acc:.4f}")
print(f"Precision  : {prec:.4f}")
print(f"Recall     : {rec:.4f}")
print(f"F1-Score   : {f1:.4f}")
print(f"ROC-AUC    : {auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test_class, pred_class_label)
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"              Non-Anemic  Anemic")
print(f"Actual Non-A      {cm[0,0]:3d}      {cm[0,1]:3d}")
print(f"Actual Anemic     {cm[1,0]:3d}      {cm[1,1]:3d}")

# ============================================================================
# Regression Metrics
# ============================================================================
print("\nREGRESSION METRICS:")
print("-" * 70)
mae = mean_absolute_error(y_test_hb, pred_hb)
mse = mean_squared_error(y_test_hb, pred_hb)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_hb, pred_hb)

print(f"MAE  : {mae:.4f} g/dL")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f} g/dL")
print(f"R²   : {r2:.4f}")

# Compare to baseline (predicting mean)
baseline_pred = np.full_like(y_test_hb, y_train_hb.mean())
baseline_mae = mean_absolute_error(y_test_hb, baseline_pred)
print(f"\nBaseline MAE (predict mean): {baseline_mae:.4f} g/dL")
print(f"Improvement: {((baseline_mae - mae) / baseline_mae * 100):.1f}%")

# ============================================================================
# Sample Predictions
# ============================================================================
print("\nSAMPLE PREDICTIONS (First 20):")
print("-" * 70)
print(f"{'#':<4} {'True':<6} {'Pred':<6} {'Prob':<7} {'True Hb':<9} {'Pred Hb':<9} {'Error':<7}")
print("-" * 70)
for i in range(min(20, len(y_test_class))):
    error = abs(y_test_hb[i] - pred_hb[i])
    print(f"{i:<4} {int(y_test_class[i]):<6} {pred_class_label[i]:<6} "
          f"{pred_class_prob[i][0]:<7.3f} {y_test_hb[i]:<9.2f} {pred_hb[i]:<9.2f} {error:<7.2f}")

print("=" * 70)
print("DONE!")
print("=" * 70)

DATA VERIFICATION
Train - Conj: (496, 1024), Text: (496, 32)
Val   - Conj: (105, 1024), Text: (105, 32)
Test  - Conj: (109, 1024), Text: (109, 32)
Class distribution - Train: [185 311]
Class distribution - Val:   [38 67]
Class distribution - Test:  [39 70]

Hemoglobin - Original range: [3.10, 15.00]
Hemoglobin - Scaled range: [-3.26, 2.05]

Projecting features to shared dimension...
After projection - Conj: (None, 1, 128), Text: (None, 1, 128)

Applying self-attention...
Applying cross-attention...
Fusion vector shape: (None, 512)

MODEL ARCHITECTURE


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Conjunctiva_Input   │ (None, 1024)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_Input          │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_41 (Dense)    │ (None, 256)       │    262,400 │ Conjunctiva_Inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_43 (Dense)    │ (None, 64)        │      2,112 │ Text_Input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_41[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_43[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_45          │ (None, 256)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_46          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_42 (Dense)    │ (None, 128)       │     32,896 │ dropout_45[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 128)       │      8,320 │ dropout_46[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_42[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_44[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 128)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 128)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conj_self_attn      │ (None, 1, 128)    │     66,048 │ reshape[0][0],    │
│ (MultiHeadAttentio… │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_self_attn      │ (None, 1, 128)    │     66,048 │ reshape_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_48          │ (None, 1, 128)    │          0 │ conj_self_attn[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_50          │ (None, 1, 128)    │          0 │ text_self_attn[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 1, 128)    │          0 │ reshape[0][0],  

 Total params: 651,906 (2.49 MB)

 Trainable params: 650,498 (2.48 MB)

...

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - Anemia_Class_accuracy: 0.4710 - Anemia_Class_auc: 0.5287 - Anemia_Class_loss: 1.0021 - Hemoglobin_loss: 3.1939 - Hemoglobin_mae: 1.3954 - loss: 3.7898

c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:946: UserWarning: You are using a softmax over axis 3 of a tensor of shape (None, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


31/31 ━━━━━━━━━━━━━━━━━━━━ 18s 83ms/step - Anemia_Class_accuracy: 0.4879 - Anemia_Class_auc: 0.5348 - Anemia_Class_loss: 0.9207 - Hemoglobin_loss: 2.6038 - Hemoglobin_mae: 1.2841 - loss: 3.6239 - val_Anemia_Class_accuracy: 0.6190 - val_Anemia_Class_auc: 0.5630 - val_Anemia_Class_loss: 0.6736 - val_Hemoglobin_loss: 1.3597 - val_Hemoglobin_mae: 0.8655 - val_loss: 3.1708 - learning_rate: 0.0010
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - Anemia_Class_accuracy: 0.5524 - Anemia_Class_auc: 0.4844 - Anemia_Class_loss: 0.7491 - Hemoglobin_loss: 1.7518 - Hemoglobin_mae: 1.0434 - loss: 3.2398 - val_Anemia_Class_accuracy: 0.6381 - val_Anemia_Class_auc: 0.5564 - val_Anemia_Class_loss: 0.7262 - val_Hemoglobin_loss: 1.2492 - val_Hemoglobin_mae: 0.8975 - val_loss: 3.0547 - learning_rate: 0.0010
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - Anemia_Class_accuracy: 0.5806 - Anemia_Class_auc: 0.5486 - Anemia_Class_loss: 0.6980 - Hemoglobin_loss: 1.5829 - Hemoglobin_mae: 1.0053 - loss: 3

c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:946: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(



CLASSIFICATION METRICS:
----------------------------------------------------------------------
Accuracy   : 0.6330
Precision  : 0.6875
Recall     : 0.7857
F1-Score   : 0.7333
ROC-AUC    : 0.5648

Confusion Matrix:
                 Predicted
              Non-Anemic  Anemic
Actual Non-A       14       25
Actual Anemic      15       55

REGRESSION METRICS:
----------------------------------------------------------------------
MAE  : 1.8832 g/dL
MSE  : 5.6011
RMSE : 2.3667 g/dL
R²   : -0.1250

Baseline MAE (predict mean): 1.8081 g/dL
Improvement: -4.2%

SAMPLE PREDICTIONS (First 20):
----------------------------------------------------------------------
#    True   Pred   Prob    True Hb   Pred Hb   Error  
----------------------------------------------------------------------
0    1      1      0.753   12.40     10.18     2.22   
1    1      1      0.992   9.90      9.69      0.21   
2    1      1      0.751   9.90      9.80      0.10   
3    0      0      0.429   12.90     10.82     2.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate, BatchNormalization, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# STEP 1: Load and normalize features (YOU ALREADY DID THIS - PERFECT!)
# ============================================================================
Conjunctiva_train = np.load("Features/conjunctiva_train_features.npy")
Text_train = np.load("Features/text_train.npy")
Conjunctiva_val = np.load('Features/conjunctiva_val_features.npy')
Text_val = np.load('Features/text_val.npy')
Conjunctiva_test = np.load('Features/conjunctiva_test_features.npy')
Text_test = np.load('Features/text_test.npy')

y_train_class = np.load("y_train_class.npy")
y_train_hb = np.load("y_train_hb.npy")
y_val_class = np.load("y_val_class.npy")
y_val_hb = np.load("y_val_hb.npy")
y_test_class = np.load("y_test_class.npy")
y_test_hb = np.load("y_test_hb.npy")

print("=" * 70)
print("DATA VERIFICATION")
print("=" * 70)
print(f"Train - Conj: {Conjunctiva_train.shape}, Text: {Text_train.shape}")
print(f"Val   - Conj: {Conjunctiva_val.shape}, Text: {Text_val.shape}")
print(f"Test  - Conj: {Conjunctiva_test.shape}, Text: {Text_test.shape}")
print(f"Class distribution - Train: {np.bincount(y_train_class.astype(int))}")
print(f"Class distribution - Val:   {np.bincount(y_val_class.astype(int))}")
print(f"Class distribution - Test:  {np.bincount(y_test_class.astype(int))}")
print("=" * 70)

# Normalize features
scaler_conj = StandardScaler()
scaler_text = StandardScaler()
scaler_hb = StandardScaler()

Palm_train = scaler_conj.fit_transform(Palm_train)
Palm_val = scaler_conj.transform(Conjunctiva_val)
Conjunctiva_test = scaler_conj.transform(Conjunctiva_test)

Text_train = scaler_text.fit_transform(Text_train)
Text_val = scaler_text.transform(Text_val)
Text_test = scaler_text.transform(Text_test)

# Normalize hemoglobin values (CRITICAL FIX!)
y_train_hb_scaled = scaler_hb.fit_transform(y_train_hb.reshape(-1, 1)).flatten()
y_val_hb_scaled = scaler_hb.transform(y_val_hb.reshape(-1, 1)).flatten()
y_test_hb_scaled = scaler_hb.transform(y_test_hb.reshape(-1, 1)).flatten()

print(f"\nHemoglobin - Original range: [{y_train_hb.min():.2f}, {y_train_hb.max():.2f}]")
print(f"Hemoglobin - Scaled range: [{y_train_hb_scaled.min():.2f}, {y_train_hb_scaled.max():.2f}]")

# ============================================================================
# STEP 2: Build Model with Self-Attention + Cross-Attention (YOUR NOVELTY!)
# ============================================================================
SHARED_DIM = 128  # Common dimension for attention

def self_attention_block(x, num_heads=4, name="self_attn"):
    """Self-attention: each modality attends to itself"""
    d_model = x.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=0.1,
        name=name
    )(x, x)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(x + attn)
    return out

def cross_attention_block(query, key_value, num_heads=4, name="cross_attn"):
    """Cross-attention: one modality attends to another"""
    d_model = query.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=0.1,
        name=name
    )(query, key_value)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(query + attn)
    return out

# Define inputs (flat features)
conj_input = Input(shape=(1024,), name="Conjunctiva_Input")
text_input = Input(shape=(32,), name="Text_Input")

# ============================================================================
# FIX #1: Project both modalities to SAME dimension before attention
# ============================================================================
print("\nProjecting features to shared dimension...")

# Conjunctiva: 1024 -> 128
conj_dense = Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(conj_input)
conj_dense = BatchNormalization()(conj_dense)
conj_dense = Dropout(0.3)(conj_dense)
conj_proj = Dense(SHARED_DIM, activation='relu')(conj_dense)
conj_proj = BatchNormalization()(conj_proj)

# Text: 32 -> 128
text_dense = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(text_input)
text_dense = BatchNormalization()(text_dense)
text_dense = Dropout(0.2)(text_dense)
text_proj = Dense(SHARED_DIM, activation='relu')(text_dense)
text_proj = BatchNormalization()(text_proj)

# Add sequence dimension for attention: (batch, 128) -> (batch, 1, 128)
conj_seq = tf.keras.layers.Reshape((1, SHARED_DIM))(conj_proj)
text_seq = tf.keras.layers.Reshape((1, SHARED_DIM))(text_proj)

print(f"After projection - Conj: {conj_seq.shape}, Text: {text_seq.shape}")

# ============================================================================
# YOUR NOVELTY: Self-Attention + Cross-Attention
# ============================================================================
print("\nApplying self-attention...")
# Each modality attends to itself
conj_self = self_attention_block(conj_seq, num_heads=4, name="conj_self_attn")
text_self = self_attention_block(text_seq, num_heads=4, name="text_self_attn")

print("Applying cross-attention...")
# Cross-modal attention (each attends to the other)
text_cross_conj = cross_attention_block(text_self, conj_self, num_heads=4, name="text_to_conj")
conj_cross_text = cross_attention_block(conj_self, text_self, num_heads=4, name="conj_to_text")

# ============================================================================
# Pool attention outputs
# ============================================================================
conj_self_pool = GlobalAveragePooling1D()(conj_self)
text_self_pool = GlobalAveragePooling1D()(text_self)
text_cross_pool = GlobalAveragePooling1D()(text_cross_conj)
conj_cross_pool = GlobalAveragePooling1D()(conj_cross_text)

# Fuse all attention outputs
fusion_vector = Concatenate()([conj_self_pool, text_self_pool, text_cross_pool, conj_cross_pool])
print(f"Fusion vector shape: {fusion_vector.shape}")

# ============================================================================
# FIX #2: Simpler fusion layers (reduce overfitting)
# ============================================================================
x_shared = Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(fusion_vector)
x_shared = BatchNormalization()(x_shared)
x_shared = Dropout(0.4)(x_shared)
x_shared = Dense(64, activation='relu')(x_shared)
x_shared = Dropout(0.3)(x_shared)

# ============================================================================
# Task-specific heads
# ============================================================================

# OPTION 1: Classification ONLY (try this first!)
print("\nBuilding CLASSIFICATION-ONLY model...")
x_class = Dense(64, activation='relu')(x_shared)
x_class = Dropout(0.3)(x_class)
x_class = Dense(32, activation='relu')(x_class)
x_class = Dropout(0.2)(x_class)
output_class = Dense(1, activation='sigmoid', name="Anemia_Class")(x_class)

model = Model(
    inputs=[conj_input, text_input],
    outputs=output_class
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), 
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)



print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
model.summary()
callbacks = [
    EarlyStopping(
        monitor='val_auc',  # Changed from val_Anemia_Class_auc
        patience=20,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    )
]

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

history = model.fit(
    [Conjunctiva_train, Text_train],
    y_train_class,  # Single output now
    validation_data=(
        [Conjunctiva_val, Text_val],
        y_val_class
    ),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)


print("\n" + "=" * 70)
print("EVALUATION ON TEST SET")
print("=" * 70)

# Get predictions
pred_class_prob = model.predict([Conjunctiva_test, Text_test], verbose=0)
pred_class_label = (pred_class_prob > 0.5).astype(int).flatten()


print("\nCLASSIFICATION METRICS:")
print("-" * 70)
acc = accuracy_score(y_test_class, pred_class_label)
prec = precision_score(y_test_class, pred_class_label, zero_division=0)
rec = recall_score(y_test_class, pred_class_label, zero_division=0)
f1 = f1_score(y_test_class, pred_class_label, zero_division=0)
auc = roc_auc_score(y_test_class, pred_class_prob)

print(f"Accuracy   : {acc:.4f}")
print(f"Precision  : {prec:.4f}")
print(f"Recall     : {rec:.4f}")
print(f"F1-Score   : {f1:.4f}")
print(f"ROC-AUC    : {auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test_class, pred_class_label)
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"              Non-Anemic  Anemic")
print(f"Actual Non-A      {cm[0,0]:3d}      {cm[0,1]:3d}")
print(f"Actual Anemic     {cm[1,0]:3d}      {cm[1,1]:3d}")

# Per-class accuracy
print(f"\nPer-class Performance:")
print(f"Non-Anemic Accuracy: {cm[0,0]/(cm[0,0]+cm[0,1]):.2%}")
print(f"Anemic Accuracy:     {cm[1,1]/(cm[1,0]+cm[1,1]):.2%}")

# Sample Predictions
print("\nSAMPLE PREDICTIONS (First 20):")
print("-" * 70)
print(f"{'#':<4} {'True':<6} {'Pred':<6} {'Prob':<8} {'Correct?':<10}")
print("-" * 70)
for i in range(min(20, len(y_test_class))):
    correct = "✓" if y_test_class[i] == pred_class_label[i] else "✗"
    print(f"{i:<4} {int(y_test_class[i]):<6} {pred_class_label[i]:<6} "
          f"{pred_class_prob[i][0]:<8.3f} {correct:<10}")


DATA VERIFICATION
Train - Conj: (496, 1024), Text: (496, 32)
Val   - Conj: (105, 1024), Text: (105, 32)
Test  - Conj: (109, 1024), Text: (109, 32)
Class distribution - Train: [185 311]
Class distribution - Val:   [38 67]
Class distribution - Test:  [39 70]

Hemoglobin - Original range: [3.10, 15.00]
Hemoglobin - Scaled range: [-3.26, 2.05]

Projecting features to shared dimension...
After projection - Conj: (None, 1, 128), Text: (None, 1, 128)

Applying self-attention...
Applying cross-attention...
Fusion vector shape: (None, 512)

Building CLASSIFICATION-ONLY model...

MODEL ARCHITECTURE


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Conjunctiva_Input   │ (None, 1024)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_Input          │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_53 (Dense)    │ (None, 256)       │    262,400 │ Conjunctiva_Inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_55 (Dense)    │ (None, 64)        │      2,112 │ Text_Input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_53[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_55[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_61          │ (None, 256)       │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_62          │ (None, 64)        │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_54 (Dense)    │ (None, 128)       │     32,896 │ dropout_61[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_56 (Dense)    │ (None, 128)       │      8,320 │ dropout_62[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_54[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_56[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1, 128)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 1, 128)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conj_self_attn      │ (None, 1, 128)    │     66,048 │ reshape_2[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_self_attn      │ (None, 1, 128)    │     66,048 │ reshape_3[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_64          │ (None, 1, 128)    │          0 │ conj_self_attn[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_66          │ (None, 1, 128)    │          0 │ text_self_attn[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 1, 128)    │          0 │ reshape_2[0][0],

 Total params: 653,953 (2.49 MB)

 Trainable params: 652,545 (2.49 MB)

...

31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5120 - auc: 0.4671 - loss: 3.2854 - precision: 0.5664 - recall: 0.6269

c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:946: UserWarning: You are using a softmax over axis 3 of a tensor of shape (None, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


31/31 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - accuracy: 0.5484 - auc: 0.4835 - loss: 3.1913 - precision: 0.6283 - recall: 0.6849 - val_accuracy: 0.6476 - val_auc: 0.5004 - val_loss: 3.0095 - val_precision: 0.6531 - val_recall: 0.9552 - learning_rate: 0.0010
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5746 - auc: 0.5206 - loss: 2.9755 - precision: 0.6295 - recall: 0.7814 - val_accuracy: 0.6571 - val_auc: 0.5463 - val_loss: 2.8163 - val_precision: 0.6505 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6129 - auc: 0.5630 - loss: 2.7584 - precision: 0.6595 - recall: 0.7910 - val_accuracy: 0.6571 - val_auc: 0.5495 - val_loss: 2.6331 - val_precision: 0.6667 - val_recall: 0.9254 - learning_rate: 0.0010
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5827 - auc: 0.5528 - loss: 2.5846 - precision: 0.6307 - recall: 0.8071 - val_accuracy: 0.6476 - val_auc: 0.5707 - val_loss: 2.4480 - val_precision:

c:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:946: UserWarning: You are using a softmax over axis 3 of a tensor of shape (32, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(



CLASSIFICATION METRICS:
----------------------------------------------------------------------
Accuracy   : 0.5413
Precision  : 0.6786
Recall     : 0.5429
F1-Score   : 0.6032
ROC-AUC    : 0.5630

Confusion Matrix:
                 Predicted
              Non-Anemic  Anemic
Actual Non-A       21       18
Actual Anemic      32       38

Per-class Performance:
Non-Anemic Accuracy: 53.85%
Anemic Accuracy:     54.29%

SAMPLE PREDICTIONS (First 20):
----------------------------------------------------------------------
#    True   Pred   Prob     Correct?  
----------------------------------------------------------------------
0    1      1      0.522    ✓         
1    1      1      0.894    ✓         
2    1      0      0.401    ✗         
3    0      0      0.395    ✓         
4    1      0      0.364    ✗         
5    1      0      0.361    ✗         
6    1      0      0.357    ✗         
7    1      0      0.350    ✗         
8    1      1      0.805    ✓         
9    0      0      

In [ ]:
# Check if labels are aligned
import pandas as pd

# Load your CSV
df = pd.read_csv("Dataset/Non_Invasive_Anemia_Dataset.csv")
train_df = df[df['Data_Type'] == 'Train']

# Sort by some ID if you have one
print("First 10 from CSV:")
print(train_df[['Age (Years)', 'GENDER', 'HB_LEVEL']].head(10))

print("\nFirst 10 labels from your processed data:")
print(y_train_class[:10])
print(y_train_hb[:10])



First 10 from CSV:
    Age (Years)  GENDER  HB_LEVEL
0           0.5  Female       9.8
1           2.0    Male       9.9
2           2.0  Female      11.1
3           1.0    Male      12.5
4           2.0    Male       9.9
5           2.0    Male      12.5
6           4.0  Female       9.9
8           3.0  Female       9.9
9           3.0  Female       9.9
11          3.0    Male      12.5

First 10 labels from your processed data:
[1. 1. 0. 0. 1. 0. 1. 1. 1. 0.]
[ 9.8  9.9 11.1 12.5  9.9 12.5  9.9  9.9  9.9 12.5]


In [97]:
# Simple model with ONLY text features (age + gender)
text_input = Input(shape=(32,))
x = Dense(64, activation='relu')(text_input)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)

model_text_only = Model(inputs=text_input, outputs=output)
model_text_only.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

history = model_text_only.fit(
    Text_train, y_train_class,
    validation_data=(Text_val, y_val_class),
    epochs=50,
    batch_size=16,
    verbose=1
)

# Test
pred = model_text_only.predict(Text_test)
auc = roc_auc_score(y_test_class, pred)
print(f"TEXT ONLY AUC: {auc:.4f}")

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.5887 - auc: 0.5196 - loss: 0.6725 - val_accuracy: 0.6381 - val_auc: 0.5198 - val_loss: 0.6595
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6149 - auc: 0.5157 - loss: 0.6705 - val_accuracy: 0.6381 - val_auc: 0.5190 - val_loss: 0.6568
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6028 - auc: 0.5019 - loss: 0.6706 - val_accuracy: 0.6381 - val_auc: 0.5542 - val_loss: 0.6537
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6169 - auc: 0.5142 - loss: 0.6672 - val_accuracy: 0.6381 - val_auc: 0.5153 - val_loss: 0.6558
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6109 - auc: 0.5179 - loss: 0.6684 - val_accuracy: 0.6381 - val_auc: 0.5408 - val_loss: 0.6524
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6169 - auc: 0.5360 - loss: 0.6616 - val_accuracy: 0.6381 - val_auc: 0.6035 - val_loss: 0.6426
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

In [99]:
# Verify text features actually contain age and gender
print("Text features stats:")
print(f"Feature 1 (should be scaled age): mean={Text_train[:, 0].mean():.3f}, std={Text_train[:, 0].std():.3f}")
print(f"Feature 2 (should be gender 0/1): mean={Text_train[:, 1].mean():.3f}, std={Text_train[:, 1].std():.3f}")
print(f"Unique values in feature 2: {np.unique(Text_train[:, 1])}")

# Your text MLP might have destroyed the information!
# Load the ORIGINAL age and gender instead:
df = pd.read_csv("Dataset/Non_Invasive_Anemia_Dataset.csv")
train_df = df[df['Data_Type'] == 'Train']

# Use RAW age and gender, not MLP features
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_df['gender_encoded'] = le.fit_transform(train_df['GENDER'])

# Create simple features
age_gender_train = train_df[['Age (Years)', 'gender_encoded']].values
scaler = StandardScaler()
age_gender_train_scaled = scaler.fit_transform(age_gender_train)

print("\nOriginal age/gender:")
print(age_gender_train[:5])

Text features stats:
Feature 1 (should be scaled age): mean=0.000, std=0.000
Feature 2 (should be gender 0/1): mean=0.000, std=1.000
Unique values in feature 2: [-2.169734   -2.1697319  -1.3445057  -1.3445038  -0.9429245  -0.5192747
 -0.51927316 -0.4260275  -0.42602676  0.09087251  0.09087288  0.3059544
  0.3059555   0.607768    0.6077695   1.1246669   1.1246673   1.1311861
  1.1660202   1.1972041   1.1972045   1.2714745   1.3382945   1.3382956
  1.3417647   1.3831158   1.4062215   1.482638    1.550566  ]

Original age/gender:
[[0.5 0. ]
 [2.  1. ]
 [2.  0. ]
 [1.  1. ]
 [2.  1. ]]


C:\Users\admin\AppData\Local\Temp\ipykernel_4804\1922042015.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['gender_encoded'] = le.fit_transform(train_df['GENDER'])


In [98]:
# Model with ONLY conjunctiva features
conj_input = Input(shape=(1024,))
x = Dense(256, activation='relu')(conj_input)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)

model_conj_only = Model(inputs=conj_input, outputs=output)
model_conj_only.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

history = model_conj_only.fit(
    Conjunctiva_train, y_train_class,
    validation_data=(Conjunctiva_val, y_val_class),
    epochs=50,
    batch_size=16,
    verbose=1
)

# Test
pred = model_conj_only.predict(Conjunctiva_test)
auc = roc_auc_score(y_test_class, pred)
print(f"CONJUNCTIVA ONLY AUC: {auc:.4f}")

Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.5625 - auc_1: 0.5621 - loss: 0.8815 - val_accuracy: 0.5143 - val_auc_1: 0.4438 - val_loss: 1.0074
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6391 - auc_1: 0.6478 - loss: 0.7380 - val_accuracy: 0.6095 - val_auc_1: 0.5035 - val_loss: 0.9732
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6694 - auc_1: 0.6704 - loss: 0.7258 - val_accuracy: 0.5714 - val_auc_1: 0.4876 - val_loss: 0.9367
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6915 - auc_1: 0.7258 - loss: 0.6297 - val_accuracy: 0.6000 - val_auc_1: 0.5587 - val_loss: 0.9719
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6875 - auc_1: 0.7337 - loss: 0.6009 - val_accuracy: 0.6190 - val_auc_1: 0.5416 - val_loss: 0.9213
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7238 - auc_1: 0.7530 - loss: 0.5829 - val_accuracy: 0.6286 - val_auc_1: 0.5450 - val_loss: 0.9288
Epoch 7/50
31/31 ━━━━━━━━━